In [ ]:
import sys
import os
import numpy as np
import time
from scipy.optimize import differential_evolution, minimize
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import rankdata
from compare import *

In [ ]:
resFolder = "./../bin"
alg_map_example = {
    #"NLSHADE_RSP": "nlshade_rsp",
    "ARRDE": "arrde",
    #"LSHADE": "lshade",
    ##"CNepsin": "lshade_cnepsin",
    #"j2020": "j2020",
    #"JSO": "jso",
    #"IMODE" : "imode", 
    #"AGSK" : "agsk",
    "RCMAES" : "rcmaes",
    "BIPOP_aCMAES" : "bipop_acmaes",
    "RDEX" : "rdex",

}

alg_map= {
    "arrde" : "ARRDE", 
    "lshade" : "LSHADE",
    "jso": "jSO",
    "lshade_cnepsin" : "LSHADE-cnEpSin", 
    "j2020": "j2020",
    "nlshade_rsp" : "NLSHADE-RSP",
    "cmaes" : "CMAES",
    "acmaes"  : "ACMAES", 
    "libcmaes_acmaes" : "L-ACMAES", 
    "libcmaes_cmaes" : "L-CMAES", 
    "rcmaes" : "RCMAES", 
    "libcmaes_abipop" : "L-ABIPOP", 
    "bipop_acmaes" : "BIPOP_aCMAES"
    
    #"IMODE" : "imode", 
    #"AGSK" : "agsk",
}


In [ ]:
ALGOS = [ "arrde",  "rcmaes",  "jso", "lshade_cnepsin", "rdex" ]
#ALGOS = ["arrde", "rcmaes",  "cmaes",  "libcmaes_cmaes",   "libcmaes_abipop"]
#ALGOS = ["arrde", "rcmaes", "bipop_acmaes", "rdex"]
YEAR = 2017
REF_ALGO = "rcmaes"
dimToYear = {
    #2017: {  10: 100000, 30: 300000, 50: 500000, 100:1000000},
    2017: {  10: 100000, 30: 300000, },
    2011: {  5: 50000, 10: 100000,},
    2020 :{5:50000, 10:1000000, 15:3000000, 20:10000000},
    2022:{10:200000, 20:1000000}
}

DIM_MAX_EVALS = dimToYear[YEAR]
summary_df, FR_df,  h2h_df = run_report(
    resFolder,
    YEAR,
    ALGOS,
    REF_ALGO,
    DIM_MAX_EVALS,
    rank_mode="ranksum",
    mwut_mode="ranksum",
    alpha=0.05,
    drop_index=1,
    mode_number=8,
    min_runs=51,
    algo_display_map=alg_map,
)

In [ ]:
ALGOS = ["lshade", "arrde", "nlshade_rsp", "j2020",  "jso", "lshade_cnepsin"]
ALGOS = ["arrde", "rcmaes", "bipop_acmaes", "cmaes", "acmaes", "libcmaes_acmaes",  "libcmaes_cmaes",   "libcmaes_abipop"]
ALGOS = ["arrde", "rcmaes", "bipop_acmaes", "rdex"]
YEAR = 2009
REF_ALGO = "rcmaes"
dimToYear = {
    2009: { 2: 20000, 5: 50000, 10: 100000, 20: 200000, 40: 400000},
}

DIM_MAX_EVALS = dimToYear[YEAR]
summary_df, FR_df,  h2h_df = run_report(
    resFolder,
    YEAR,
    ALGOS,
    REF_ALGO,
    DIM_MAX_EVALS,
    rank_mode="ranksum",
    mwut_mode="ranksum",
    alpha=0.05,
    drop_index=None,
    mode_number=8,
    algo_display_map=alg_map,
    min_runs=0
)

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "RCMAES"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key=reference,         # choose reference algorithm name
        year=2009,               # ← no longer hardcoded
        maxevals=200000,
        n_dim=20,
        alpha=0.05,
        drop_index=None,
        n_problems=24,
        mwut_mode="ranksum",
        mode_number=8
    )

In [ ]:
MAXEVALS_LIST = [2000, 20000, 200000, 2000000, 20000000]
ALGOS = ["arrde", "rcmaes", "bipop_acmaes", "j2020", "nlshade_rsp", "lshade","libcmaes_abipop", "rdex"]
ALGOS = [ "rcmaes", "bipop_acmaes", "libcmaes_abipop", "arrde", "rdex"]
YEAR = 2009
REF_ALGO = "rcmaes"
DIM = 20
MIN_RUNS = 21
SOLVED_ATOL = 1e-8
drop_index = None
mode_number = 8

# Solve a function if its final value matches the known BBOB optimum up to SOLVED_ATOL.
gopt = bbob2009_global_optimum(DIM, 24)
if gopt is None:
    raise RuntimeError("BBOB2009 optima are unavailable.")

def resolve_result_file(res_folder, year, algo, dim, maxevals):
    fname = f"results_{year}_{algo}_{dim}_{maxevals}.txt"
    path = os.path.join(res_folder, fname)
    if os.path.exists(path):
        return path
    directory = os.path.dirname(path)
    target = os.path.basename(path).lower()
    if not os.path.isdir(directory):
        raise FileNotFoundError(f"Directory not found: {directory}")
    for entry in os.listdir(directory):
        if entry.lower() == target:
            return os.path.join(directory, entry)
    raise FileNotFoundError(f"File not found: {path}")

records = []
for maxevals in MAXEVALS_LIST:
    dimToYear = {YEAR: {DIM: maxevals}}
    summary_df, FR_df, h2h_df = run_report(
        resFolder,
        YEAR,
        ALGOS,
        REF_ALGO,
        dimToYear[YEAR],
        rank_mode="ranksum",
        mwut_mode="ranksum",
        alpha=0.05,
        drop_index=drop_index,
        mode_number=mode_number,
        algo_display_map=alg_map,
        min_runs=MIN_RUNS,
    )
    for algo, row in summary_df.iterrows():
        filepath = resolve_result_file(resFolder, YEAR, algo, DIM, maxevals)
        data = np.atleast_2d(np.loadtxt(filepath, delimiter="\t"))
        if data.shape[0] < MIN_RUNS:
            raise ValueError(f"{algo} @ {DIM}D has only {data.shape[0]} runs, need at least {MIN_RUNS}.")
        data = data[:MIN_RUNS]
        solved = np.isclose(np.abs((data - gopt)/gopt), 0.0, atol=SOLVED_ATOL, rtol=0.0)
        solved_fraction = float(np.mean(solved))
        records.append(
            {
                "MaxEvals": maxevals,
                "Algorithm": algo,
                "DisplayName": alg_map.get(algo, algo),
                "SE": row["SE_weighted"],
                "SR": row["SR_weighted"],
                "SolvedFraction": solved_fraction,
            }
        )

trend_df = pd.DataFrame(records)
trend_df

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharex=True)
plot_specs = [
     (axes[0], "SE", "Average rel errors", None, None),
    (axes[1], "SR", "Average Friedman ranks", None, None),
    (axes[2], "SolvedFraction", "Fraction of solved problems", 0.0, 1.0),
]
for ax, metric, title, ymin, ymax in plot_specs:
    for algo in ALGOS:
        algo_df = trend_df[trend_df["Algorithm"] == algo].sort_values("MaxEvals")
        ax.plot(
            np.log10(algo_df["MaxEvals"]/DIM),
            algo_df[metric],
            marker="o",
            linewidth=2,
            label=alg_map.get(algo, algo),
        )
    ax.set_title("BBOB 2009 @20D 24 functions")
    ax.set_xlabel(r"$\log_{10}(N_{\text{max}} / D)$")
    ax.set_ylabel(title)
    ax.set_xticks(np.log10(np.array(MAXEVALS_LIST) / DIM))
    ax.set_xscale("linear")
    if ymin is not None and ymax is not None:
        ax.set_ylim(ymin, ymax)
    ax.grid(True, linestyle="--", alpha=0.4)
    if metric == "SE":
        ax.set_yscale("log")

axes[0].legend(loc="best")
plt.tight_layout()
fig.savefig("2009_se_sr_solved_vs_maxevals.pdf", dpi=300, bbox_inches="tight")


In [ ]:
DIM = 10
MAXEVALS_LIST = {
    50 : [5000, 20000, 50000, 500000,  2000000, 5000000], 
    30 : [3000, 10000, 30000, 100000, 300000, 1000000, 3000000],
    10 : [1000, 5000, 10000, 50000, 100000, 1000000, 10000000], 
    100 : [10000,100000, 500000, 1000000]
}[DIM]
ALGOS = {
    50: ["arrde", "rcmaes",  "rdex", "bipop_acmaes"],
    30: ["arrde", "rcmaes",  "rdex", "bipop_acmaes"], #["arrde", "rcmaes", "bipop_acmaes", "cmaes", "libcmaes_abipop", "rdex"]
    10: ["arrde", "rcmaes",  "rdex", "bipop_acmaes"], 
    100: [ "rcmaes",  "rdex", "arrde"]
}[DIM]

#ALGOS = ["arrde", "rcmaes", "bipop_acmaes", "cmaes", "libcmaes_cmaes", "libcmaes_abipop", "rdex"]
YEAR = 2017
REF_ALGO = "rcmaes"
MIN_RUNS = 21
SOLVED_ATOL = 1e-8
drop_index = None
mode_number = 8

# Solve a function if its final value matches the known BBOB optimum up to SOLVED_ATOL.
gopt = 100*np.array([i+1 for i in range(30)])

def resolve_result_file(res_folder, year, algo, dim, maxevals):
    fname = f"results_{year}_{algo}_{dim}_{maxevals}.txt"
    path = os.path.join(res_folder, fname)
    if os.path.exists(path):
        return path
    directory = os.path.dirname(path)
    target = os.path.basename(path).lower()
    if not os.path.isdir(directory):
        raise FileNotFoundError(f"Directory not found: {directory}")
    for entry in os.listdir(directory):
        if entry.lower() == target:
            return os.path.join(directory, entry)
    raise FileNotFoundError(f"File not found: {path}")

records = []
for maxevals in MAXEVALS_LIST:
    dimToYear = {YEAR: {DIM: maxevals}}
    summary_df, FR_df, h2h_df = run_report(
        resFolder,
        YEAR,
        ALGOS,
        REF_ALGO,
        dimToYear[YEAR],
        rank_mode="ranksum",
        mwut_mode="ranksum",
        alpha=0.05,
        drop_index=drop_index,
        mode_number=mode_number,
        algo_display_map=alg_map,
        min_runs=MIN_RUNS,
    )
    for algo, row in summary_df.iterrows():
        filepath = resolve_result_file(resFolder, YEAR, algo, DIM, maxevals)
        data = np.atleast_2d(np.loadtxt(filepath, delimiter="\t"))
        if data.shape[0] < MIN_RUNS:
            raise ValueError(f"{algo} @ {DIM}D has only {data.shape[0]} runs, need at least {MIN_RUNS}.")
        data = data[:MIN_RUNS]
        run_success = np.abs((data - gopt) / gopt) <= SOLVED_ATOL
        function_solved = np.mean(run_success, axis=0) >= 0.5
        solved_fraction = float(np.mean(function_solved))
        records.append(
            {
                "MaxEvals": maxevals,
                "Algorithm": algo,
                "DisplayName": alg_map.get(algo, algo),
                "SE": row["SE_weighted"],
                "SR": row["SR_weighted"],
                "SolvedFraction": solved_fraction,
            }
        )

trend_df = pd.DataFrame(records)
trend_df

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharex=True)
plot_specs = [
    (axes[0], "SE", "Average rel errors", None, None),
    (axes[1], "SR", "Average Friedman ranks", None, None),
    (axes[2], "SolvedFraction", "Fraction of solved problems", 0.0, 1.0),
]
for ax, metric, title, ymin, ymax in plot_specs:
    for algo in ALGOS:
        algo_df = trend_df[trend_df["Algorithm"] == algo].sort_values("MaxEvals")
        ax.plot(
            np.log10(algo_df["MaxEvals"]/DIM),
            algo_df[metric],
            marker="o",
            linewidth=2,
            label=alg_map.get(algo, algo),
        )
    ax.set_title(f"CEC2017 @{DIM}D 30 functions")
    ax.set_xlabel(r"$\log_{10}(N_{\text{max}} / D)$")
    ax.set_ylabel(title)
    ax.set_xticks(np.log10(np.array(MAXEVALS_LIST) / DIM))
    ax.set_xscale("linear")
    if ymin is not None and ymax is not None:
        ax.set_ylim(ymin, ymax)
    ax.grid(True, linestyle="--", alpha=0.4)
    if metric == "SE":
        ax.set_yscale("log")

axes[0].legend(loc="best")
plt.tight_layout()
fig.savefig(f"2017_se_sr_solved_vs_maxevals_{{DIM}}D.pdf", dpi=300, bbox_inches="tight")


In [ ]:
ALGOS = ["lshade", "arrde", "nlshade_rsp", "j2020",  "jso", "lshade_cnepsin", "rcmaes"]
YEAR = 2011
REF_ALGO = "arrde"
MULTIPLIER = 10000
dimToYear = {
    2017: {10: 10*MULTIPLIER, 30: 30*MULTIPLIER, 50: 50*MULTIPLIER, 100: 100*MULTIPLIER},
    2014: {10: 10*MULTIPLIER, 30: 30*MULTIPLIER, 50: 50*MULTIPLIER, 100: 100*MULTIPLIER},
    2020: {5: 50000, 10: 1000000, 15: 3000000, 20: 10000000},
    2022: {10: 200000, 20: 1000000},
    2011 :{ 5:50000, 10:100000, 15:150000}, 
    2019 : {10:100000000}
}

DIM_MAX_EVALS = dimToYear[YEAR]
summary_df, FR_df,  h2h_df = run_report(
    resFolder,
    YEAR,
    ALGOS,
    REF_ALGO,
    DIM_MAX_EVALS,
    rank_mode="ranksum",
    mwut_mode="ranksum",
    alpha=0.05,
    drop_index=None,
    mode_number=9,
    algo_display_map=alg_map,
)

# Scoring the Performance of DE Variants on Multiple CEC Problems (Higher is better. Maximum score =100)

### CEC 2014, 2017, 2019, 2020, 2022

In [ ]:
from compare import compute_scores_vs_multiplier, plot_score_trends

algos = ["LSHADE",  "ARRDE", "LSHADE_cnEpSin", "jSO", "NLSHADE_RSP", "j2020"]
year = 2022
ref_algo = "ARRDE"

multiplier_list = [500, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

base_dim_templates = {
    2014: {10: 10, 30: 30, 50: 50, 100: 100},
    2017: {10: 10, 30: 30, 50: 50 , 100: 100},
    2019: {10: 10},
    2020: {5: 5, 10: 10, 15: 15, 20: 20},
    2022: {10: 10, 20: 20},
}

base_template = base_dim_templates.get(year)
if base_template is None:
    raise ValueError(f"No base dim template configured for year {year}")

score_df = compute_scores_vs_multiplier(
    res_folder=resFolder,
    year=year,
    algos=algos,
    ref_algo=ref_algo,
    multipliers=multiplier_list,
    base_dim_template=base_template,
    algo_display_map=alg_map,
    rank_mode="ranksum",
    mwut_mode="ranksum",
    alpha=0.05,
    mode_number=8,
    drop_index=None,
)

plot_score_trends(
    score_df,
    #metrics=["Score1(SE)", "Score2(SR)", "FinalScore"],
    metrics=[ "FinalScore"],
    xlabel=r"$N_{max}/D$",
    legend_labels=alg_map,
    save_as="2022.pdf", 
     figsize=(6, 4.5),
)


# CEC2011

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2011,               # ← no longer hardcoded
        maxevals=50000,
        n_dim=5,
        alpha=0.05,
        drop_index=None,
        n_problems=22,
        mwut_mode="ranksum",
        mode_number=6
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        resFolder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2011,               # ← no longer hardcoded
        maxevals=100000,
        n_dim=10,
        alpha=0.05,
        drop_index=None,
        n_problems=22,
        mwut_mode="ranksum",
        mode_number=3
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2011,               # ← no longer hardcoded
        maxevals=150000,
        n_dim=15,
        alpha=0.05,
        drop_index=None,
        n_problems=22,
        mwut_mode="ranksum",
        mode_number=7
    )

# CEC 2017

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        resFolder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2017,               # ← no longer hardcoded
        maxevals=100000,
        n_dim=10,
        alpha=0.05,
        drop_index=1,
        n_problems=30,
        mwut_mode="ranksum",
        mode_number=3
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        resFolder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2017,               # ← no longer hardcoded
        maxevals=300000,
        n_dim=30,
        alpha=0.05,
        n_problems=30,
        drop_index=1,
        mwut_mode="ranksum", 
        mode_number=3
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        resFolder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2017,               # ← no longer hardcoded
        maxevals=500000,
        n_dim=50,
        alpha=0.05,
        n_problems=30,
        drop_index=1,
        mwut_mode="ranksum"
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        resFolder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2017,               # ← no longer hardcoded
        maxevals=1000000,
        n_dim=100,
        alpha=0.05,
        n_problems=30,
        drop_index=1,
        mwut_mode="ranksum"
    )

# CEC 2020

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2020,               # ← no longer hardcoded
        maxevals=50000,
        n_dim=5,
        alpha=0.05,
        n_problems=10,
        drop_index=None,
        mwut_mode="ranksum",
        mode_number=1,
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2020,               # ← no longer hardcoded
        maxevals=1000000,
        n_dim=10,
        alpha=0.05,
        n_problems=10,
        drop_index=None,
        mwut_mode="ranksum", 
        mode_number=1,
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2020,               # ← no longer hardcoded
        maxevals=3000000,
        n_dim=15,
        alpha=0.05,
        n_problems=10,
        drop_index=None,
        mwut_mode="ranksum", 
        mode_number=1,
    )

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2020,               # ← no longer hardcoded
        maxevals=10000000,
        n_dim=20,
        alpha=0.05,
        n_problems=10,
        drop_index=None,
        mwut_mode="ranksum", 
        mode_number=1,
    )

# CEC 2022


## 10D

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2022,               # ← no longer hardcoded
        maxevals=200000,
        n_dim=10,
        alpha=0.05,
        n_problems=12,
        drop_index=None,
        mwut_mode="mean"
    )

## 20D


In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2022,               # ← no longer hardcoded
        maxevals=1000000,
        n_dim=20,
        alpha=0.05,
        n_problems=12,
        drop_index=None,
        mwut_mode="ranksum"
    )

# CEC2019

In [ ]:
# choose reference algorithm by its display name (not hardcoded)
reference = "ARRDE"

# call (assumes resFolder, goptimum_cec17 and files exist in your environment)
result = analyze_results(
        res_folder=resFolder,     # your results folder path
        alg_map=alg_map_example,
        ref_key="ARRDE",         # choose reference algorithm name
        year=2019,               # ← no longer hardcoded
        maxevals=100000000,
        n_dim=10,
        alpha=0.05,
        n_problems=10,
        drop_index=None,
        mwut_mode="ranksum", 
        mode_number=8,
    )

In [ ]:
filling_table, scores, total = compute_cec100_scores("2019/results_2019_j2020_10_100000000.txt")

print("\n=== CEC 2019 100-digit Challenge Results ===\n")
print(filling_table)
print("\nTotal Score: {:.2f}".format(total))

In [ ]:
filling_table, scores, total = compute_cec100_scores("2019/results_2019_lshade_cnepsin_10_100000000.txt")

print("\n=== CEC 2019 100-digit Challenge Results ===\n")
print(filling_table)
print("\nTotal Score: {:.3f}".format(total))